# Strategy comparison

uniform_swing vs reform_threat_consolidation. List flips; chart national-total deltas.

In [ ]:
import os
from pathlib import Path

def _find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not find repo root (no pyproject.toml in cwd or any parent)")

os.chdir(_find_repo_root())

def _latest_snapshot_hash() -> str:
    """Return the content hash of the most recent snapshot. Filenames are
    YYYY-MM-DD__v<schema>__<hash>.sqlite, so lexical sort = chronological."""
    snaps = sorted(Path("data/snapshots").glob("*.sqlite"))
    if not snaps:
        raise FileNotFoundError("no snapshots in data/snapshots/; run /snapshot first")
    return snaps[-1].stem.split("__")[-1]

def _pick_prediction(strategy_marker: str, label: str) -> Path:
    """Select the prediction file for the LATEST snapshot whose name contains
    strategy_marker AND label. Prediction filenames are
    <snap_hash>__<strategy>__<config_hash>__<label>.sqlite. Fails loud if 0 or
    >1 files match — the previous sorted(glob)[-1] silently picked an
    alphabetically-last file, which became nondeterministic once sweep_* runs
    or older snapshots' predictions shared the directory."""
    snap_hash = _latest_snapshot_hash()
    pred_dir = Path("data/predictions")
    matches = [
        p for p in pred_dir.glob(f"{snap_hash}__*{strategy_marker}*.sqlite")
        if label in p.name
    ]
    if not matches:
        raise FileNotFoundError(
            f"no prediction in {pred_dir} for snapshot {snap_hash} matching "
            f"strategy={strategy_marker!r} label={label!r}; run /predict first"
        )
    if len(matches) > 1:
        names = sorted(p.name for p in matches)
        raise RuntimeError(
            f"multiple predictions for snapshot {snap_hash} match "
            f"strategy={strategy_marker!r} label={label!r}: {names}; "
            f"remove duplicates or pass a more specific label"
        )
    return matches[0]

In [ ]:
from prediction_engine.analysis.flips import compute_flips
from prediction_engine.sqlite_io import read_prediction_national

us_run  = _pick_prediction("uniform_swing", "baseline_us")
rtc_run = _pick_prediction("reform_threat_consolidation", "baseline_rtc")
flips = compute_flips(us_run, rtc_run)
flips.head(20)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
nat_us  = read_prediction_national(us_run)
nat_rtc = read_prediction_national(rtc_run)
us_overall  = nat_us [nat_us ["scope"] == "overall"].set_index("party")["seats"]
rtc_overall = nat_rtc[nat_rtc["scope"] == "overall"].set_index("party")["seats"]
pd.DataFrame({"uniform_swing": us_overall, "reform_threat": rtc_overall}).plot.bar(figsize=(8, 4))
plt.ylabel("Seats")
plt.title("National totals: uniform_swing vs reform_threat_consolidation")
plt.show()

If reform_threat trims Reform seats vs uniform_swing while raising Lab/LD/Green/Plaid/SNP, the consolidation strategy is firing as expected.